In [1]:
# ==============================
# INSTALL REQUIRED LIBRARIES
# ==============================

!pip install -q wandb
!pip install -q librosa
!pip install -q tqdm

In [2]:
# =========================================
# IMPORTS + DEVICE SETUP + RANDOM SEED
# =========================================

import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import librosa
import librosa.display

from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split

from tqdm import tqdm
import wandb

import warnings
warnings.filterwarnings("ignore")

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [3]:
# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
# -----------------------------
# Seed for reproducibility
# -----------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [5]:
# =========================================
# CONFIGURATION
# =========================================

class CFG:
    
    # Paths (UPDATE DATASET PATH IF NEEDED)
    DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
    GENRE_STEMS = os.path.join(DATA_ROOT, "genres_stems")
    MASHUP_DIR = os.path.join(DATA_ROOT, "mashups")
    TEST_CSV = os.path.join(DATA_ROOT, "test.csv")
    SAMPLE_SUB = os.path.join(DATA_ROOT, "sample_submission.csv")
    
    # Audio
    SAMPLE_RATE = 22050
    DURATION = 30      # seconds
    N_MFCC = 40
    
    # Training
    BATCH_SIZE = 16
    EPOCHS = 10
    LR = 1e-3
    
    # Misc
    NUM_CLASSES = 10
    SEED = 42
    MODEL_SAVE_PATH = "lstm_mfcc_model.pth"
    
    TRAIN_MODE = True   # Set False when submitting


print("Config loaded successfully.")

Config loaded successfully.


In [6]:
# =========================================
# LABEL MAPPING + TRAIN/VAL SPLIT
# =========================================

GENRES = [
    "blues", "classical", "country", "disco", "hiphop",
    "jazz", "metal", "pop", "reggae", "rock"
]

label2idx = {genre: idx for idx, genre in enumerate(GENRES)}
idx2label = {idx: genre for genre, idx in label2idx.items()}

print("Label mapping:", label2idx)


# Collect all song folders (each song contains 4 stems)
all_songs = []

for genre in GENRES:
    genre_path = os.path.join(CFG.GENRE_STEMS, genre)
    song_folders = os.listdir(genre_path)
    
    for song in song_folders:
        song_path = os.path.join(genre_path, song)
        all_songs.append((song_path, label2idx[genre]))

print("Total songs found:", len(all_songs))


# Train/Validation split
train_songs, val_songs = train_test_split(
    all_songs,
    test_size=0.2,
    stratify=[label for _, label in all_songs],
    random_state=CFG.SEED
)

print("Train samples:", len(train_songs))
print("Validation samples:", len(val_songs))

Label mapping: {'blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4, 'jazz': 5, 'metal': 6, 'pop': 7, 'reggae': 8, 'rock': 9}
Total songs found: 1000
Train samples: 800
Validation samples: 200


In [7]:
# =========================================
# LOAD ESC-50 NOISE FILE PATHS
# =========================================

ESC_AUDIO_DIR = os.path.join(CFG.DATA_ROOT, "ESC-50-master", "audio")

noise_files = [
    os.path.join(ESC_AUDIO_DIR, f)
    for f in os.listdir(ESC_AUDIO_DIR)
    if f.endswith(".wav")
]

print("Total noise files found:", len(noise_files))

Total noise files found: 2000


In [8]:
# =========================================
# UPDATED DATASET WITH NOISE AUGMENTATION
# =========================================

class MashupDataset(Dataset):
    def __init__(self, song_list, train=True):
        self.song_list = song_list
        self.train = train
        self.sample_rate = CFG.SAMPLE_RATE
        self.duration = CFG.DURATION
        self.n_mfcc = CFG.N_MFCC
        self.max_length = self.sample_rate * self.duration

    def load_audio(self, path):
        audio, sr = librosa.load(path, sr=self.sample_rate)
        
        # Pad or trim
        if len(audio) > self.max_length:
            audio = audio[:self.max_length]
        else:
            pad_length = self.max_length - len(audio)
            audio = np.pad(audio, (0, pad_length))
            
        return audio

    def add_noise(self, audio):
        noise_path = random.choice(noise_files)
        noise = self.load_audio(noise_path)
        
        # Random noise strength
        alpha = random.uniform(0.01, 0.05)
        
        audio = audio + alpha * noise
        return audio

    def __len__(self):
        return len(self.song_list)

    def __getitem__(self, idx):
        song_path, label = self.song_list[idx]
        
        drums = self.load_audio(os.path.join(song_path, "drums.wav"))
        vocals = self.load_audio(os.path.join(song_path, "vocals.wav"))
        bass = self.load_audio(os.path.join(song_path, "bass.wav"))
        other = self.load_audio(os.path.join(song_path, "other.wav"))
        
        if self.train:
            # Random gain scaling per stem
            drums *= random.uniform(0.7, 1.3)
            vocals *= random.uniform(0.7, 1.3)
            bass *= random.uniform(0.7, 1.3)
            other *= random.uniform(0.7, 1.3)
        
        mixed_audio = (drums + vocals + bass + other) / 4.0
        
        if self.train:
            # 70% chance to add noise
            if random.random() < 0.7:
                mixed_audio = self.add_noise(mixed_audio)
        
        # Extract MFCC
        mfcc = librosa.feature.mfcc(
            y=mixed_audio,
            sr=self.sample_rate,
            n_mfcc=self.n_mfcc
        )
        
        mfcc = (mfcc - np.mean(mfcc)) / (np.std(mfcc) + 1e-6)
        
        mfcc = torch.tensor(mfcc, dtype=torch.float32)
        mfcc = mfcc.unsqueeze(0)
        
        label = torch.tensor(label, dtype=torch.long)
        
        return mfcc, label


In [9]:
dataset_test = MashupDataset(train_songs)
sample_mfcc, sample_label = dataset_test[0]

print("MFCC shape:", sample_mfcc.shape)
print("Label:", sample_label)

MFCC shape: torch.Size([1, 40, 1292])
Label: tensor(5)


In [10]:
train_dataset = MashupDataset(train_songs, train=True)
val_dataset = MashupDataset(val_songs, train=False)

train_loader = DataLoader(train_dataset, batch_size=CFG.BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [11]:
# # =========================================
# # CNN MODEL (MFCC INPUT)
# # =========================================

# class CNN_MFCC(nn.Module):
#     def __init__(self, num_classes=CFG.NUM_CLASSES):
#         super(CNN_MFCC, self).__init__()
        
#         self.features = nn.Sequential(
#             nn.Conv2d(1, 16, kernel_size=3, padding=1),
#             nn.BatchNorm2d(16),
#             nn.ReLU(),
#             nn.MaxPool2d(2),
            
#             nn.Conv2d(16, 32, kernel_size=3, padding=1),
#             nn.BatchNorm2d(32),
#             nn.ReLU(),
#             nn.MaxPool2d(2),
            
#             nn.Conv2d(32, 64, kernel_size=3, padding=1),
#             nn.BatchNorm2d(64),
#             nn.ReLU(),
#             nn.MaxPool2d(2),
#         )
        
#         # We will compute this dynamically
#         self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        
#         self.classifier = nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(64, 128),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(128, num_classes)
#         )
        
#     def forward(self, x):
#         x = self.features(x)
#         x = self.global_pool(x)
#         x = self.classifier(x)
#         return x


# # Initialize model
# model = CNN_MFCC().to(device)

# print(model)

CNN_MFCC(
  (features): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (global_pool): AdaptiveAvgPool2d(output_size=(1, 1))
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=64, out_featur

In [13]:
# =========================================
# LSTM MODEL (MFCC SEQUENCE)
# =========================================

class LSTM_MFCC(nn.Module):
    def __init__(self, input_size=CFG.N_MFCC, hidden_size=128, num_layers=2, num_classes=CFG.NUM_CLASSES):
        super(LSTM_MFCC, self).__init__()
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )
        
        self.dropout = nn.Dropout(0.3)
        
        self.fc = nn.Linear(hidden_size * 2, num_classes)
        
    def forward(self, x):
        # x shape: (batch, 1, 40, T)
        
        x = x.squeeze(1)          # (batch, 40, T)
        x = x.permute(0, 2, 1)    # (batch, T, 40)
        
        output, _ = self.lstm(x)
        
        # Take last time step
        last_output = output[:, -1, :]
        
        out = self.dropout(last_output)
        out = self.fc(out)
        
        return out

In [14]:
model = LSTM_MFCC().to(device)
print(model)

LSTM_MFCC(
  (lstm): LSTM(40, 128, num_layers=2, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=10, bias=True)
)


In [12]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("DLGENAI_WANDB_API_KEY")
os.environ["WANDB_API_KEY"] = secret_value_0
wandb.login()

wandb: Currently logged in as: sharibahmad (sharibahmad-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [15]:
# =========================================
# LOSS, OPTIMIZER, WANDB SETUP (SAFE MODE)
# =========================================

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=CFG.LR)


import wandb

if CFG.TRAIN_MODE:
    wandb.init(
        project="24f2001786-t12026",
        name="lstm_mfcc_model",
        config={
            "epochs": CFG.EPOCHS,
            "batch_size": CFG.BATCH_SIZE,
            "lr": CFG.LR,
            "n_mfcc": CFG.N_MFCC
        },
        reinit=True
    )

print("WandB safely initialized.")

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


WandB safely initialized.


In [16]:
# =========================================
# TRAINING + VALIDATION LOOP (WITH WANDB)
# =========================================

from sklearn.metrics import f1_score, accuracy_score

best_f1 = 0.0

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    loop = tqdm(loader, desc="Training", leave=False)
    
    for inputs, labels in loop:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())
    
    epoch_loss = total_loss / len(loader)
    epoch_f1 = f1_score(all_labels, all_preds, average="macro")
    epoch_acc = accuracy_score(all_labels, all_preds)
    
    return epoch_loss, epoch_f1, epoch_acc


def validate(model, loader):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    loop = tqdm(loader, desc="Validation", leave=False)
    
    with torch.no_grad():
        for inputs, labels in loop:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            loop.set_postfix(loss=loss.item())
    
    epoch_loss = total_loss / len(loader)
    epoch_f1 = f1_score(all_labels, all_preds, average="macro")
    epoch_acc = accuracy_score(all_labels, all_preds)
    
    return epoch_loss, epoch_f1, epoch_acc


# =========================
# TRAINING START
# =========================

if CFG.TRAIN_MODE:
    
    for epoch in range(CFG.EPOCHS):
        print(f"\nEpoch {epoch+1}/{CFG.EPOCHS}")
        
        train_loss, train_f1, train_acc = train_one_epoch(model, train_loader)
        val_loss, val_f1, val_acc = validate(model, val_loader)
        
        print(f"Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val   Loss: {val_loss:.4f} | Val   F1: {val_f1:.4f} | Val   Acc: {val_acc:.4f}")
        
        # 🔥 WandB Logging
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_f1": train_f1,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_f1": val_f1,
            "val_acc": val_acc
        })
        
        # Save best model
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), CFG.MODEL_SAVE_PATH)
            wandb.save(CFG.MODEL_SAVE_PATH)
            print("✅ Best model saved!")


Epoch 1/10


Train Loss: 2.2746 | Train F1: 0.1144 | Train Acc: 0.1700
Val   Loss: 2.2812 | Val   F1: 0.0357 | Val   Acc: 0.1050
✅ Best model saved!

Epoch 2/10


Train Loss: 2.1866 | Train F1: 0.1249 | Train Acc: 0.1750
Val   Loss: 2.0635 | Val   F1: 0.1483 | Val   Acc: 0.2600
✅ Best model saved!

Epoch 3/10


Train Loss: 1.9679 | Train F1: 0.2262 | Train Acc: 0.2975
Val   Loss: 1.8701 | Val   F1: 0.3231 | Val   Acc: 0.3550
✅ Best model saved!

Epoch 4/10


Train Loss: 1.8404 | Train F1: 0.2857 | Train Acc: 0.3287
Val   Loss: 1.8088 | Val   F1: 0.2345 | Val   Acc: 0.3250

Epoch 5/10


...

Train Loss: 1.6938 | Train F1: 0.3605 | Train Acc: 0.3875
Val   Loss: 1.6990 | Val   F1: 0.3498 | Val   Acc: 0.4150
✅ Best model saved!

Epoch 7/10


Train Loss: 1.6180 | Train F1: 0.3858 | Train Acc: 0.4163
Val   Loss: 1.7393 | Val   F1: 0.3389 | Val   Acc: 0.3600

Epoch 8/10


Train Loss: 1.6005 | Train F1: 0.3939 | Train Acc: 0.4175
Val   Loss: 1.6511 | Val   F1: 0.3673 | Val   Acc: 0.3750
✅ Best model saved!

Epoch 9/10


Train Loss: 1.5198 | Train F1: 0.4007 | Train Acc: 0.4375
Val   Loss: 1.6460 | Val   F1: 0.3848 | Val   Acc: 0.3850
✅ Best model saved!

Epoch 10/10


Train Loss: 1.4641 | Train F1: 0.4654 | Train Acc: 0.4788
Val   Loss: 1.5427 | Val   F1: 0.3751 | Val   Acc: 0.3850


In [18]:
# =========================================
# LOAD BEST MODEL FOR INFERENCE
# =========================================

MODEL_PATH = "/kaggle/input/models/genrede/lstm-mfcc-model/pytorch/default/1/lstm_mfcc_model.pth"

model = LSTM_MFCC().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

print("Model loaded successfully.")


Model loaded successfully.


In [19]:
# =========================================
# TEST DATASET (CORRECT VERSION)
# =========================================

class TestDataset(Dataset):
    def __init__(self, test_csv_path, data_root):
        self.test_df = pd.read_csv(test_csv_path)
        self.data_root = data_root
        self.sample_rate = CFG.SAMPLE_RATE
        self.duration = CFG.DURATION
        self.n_mfcc = CFG.N_MFCC
        self.max_length = self.sample_rate * self.duration
        
    def load_audio(self, path):
        audio, sr = librosa.load(path, sr=self.sample_rate)
        
        if len(audio) > self.max_length:
            audio = audio[:self.max_length]
        else:
            pad_length = self.max_length - len(audio)
            audio = np.pad(audio, (0, pad_length))
            
        return audio
    
    def __len__(self):
        return len(self.test_df)
    
    def __getitem__(self, idx):
        row = self.test_df.iloc[idx]
        file_id = row["id"]
        filename = row["filename"]
        
        audio_path = os.path.join(CFG.DATA_ROOT, filename)
        audio = self.load_audio(audio_path)
        
        mfcc = librosa.feature.mfcc(
            y=audio,
            sr=self.sample_rate,
            n_mfcc=self.n_mfcc
        )
        
        mfcc = (mfcc - np.mean(mfcc)) / (np.std(mfcc) + 1e-6)
        
        mfcc = torch.tensor(mfcc, dtype=torch.float32).unsqueeze(0)
        
        return mfcc, file_id

In [20]:
test_dataset = TestDataset(CFG.TEST_CSV, CFG.DATA_ROOT)

test_loader = DataLoader(
    test_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Test samples:", len(test_dataset))

Test samples: 3020


In [21]:
# =========================================
# INFERENCE (FIXED ID FORMAT)
# =========================================

predictions = []
ids = []

model.eval()

with torch.no_grad():
    for inputs, file_ids in tqdm(test_loader, desc="Inference"):
        inputs = inputs.to(device)
        
        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1)
        
        predictions.extend(preds.cpu().numpy())
        
        # Convert tensor IDs to normal Python values
        ids.extend([int(i) for i in file_ids])


# Convert indices to genre names
predicted_genres = [idx2label[p] for p in predictions]

submission = pd.DataFrame({
    "id": ids,
    "genre": predicted_genres
})

submission.to_csv("/kaggle/working/submission.csv", index=False)
print("Saved at:", os.path.exists("/kaggle/working/submission.csv"))

print("Submission file created successfully!")
submission.head()

Inference: 100%|██████████| 189/189 [03:10<00:00,  1.01s/it]

Saved at: True
Submission file created successfully!


,id,genre
0,1,reggae
1,2,classical
2,3,reggae
3,4,reggae
4,5,reggae
